# Screening

In [ ]:
pip install asreview

In [ ]:
import pandas as pd

df = pd.read_csv('/content/list_final.csv', encoding='latin-1')
print(f"Total papers: {len(df)}")

relevant_titles_keywords = [
    "Adherence to a Nurse-Driven Feeding Protocol in a Pediatric Intensive Care Unit",          # Cunningham 2017
    "Impact of a nurse-led feeding protocol in a pediatric intensive care unit",              # Ang 2016
    "An evaluation of enteral feeding practices in critically ill children",                 # Tume 2017
    "Impact of Nutrition Support Team in Achieving Target Calories in Children Admitted in Pediatric Intensive Care Unit",                  # Zeeshan 2022
    "Development and implementation of a feeding protocol for infants in a pediatric cardiac intensive care unit",              # Um 2016
    "Bedside Postpyloric Tube Placement and Enteral Nutrition Delivery in the Pediatric Intensive Care Unit",                     # Turner 2020
]

df['label_included'] = None

for keyword in relevant_titles_keywords:
    mask = df['title'].str.contains(keyword, case=False, na=False)
    df.loc[mask, 'label_included'] = 1
    matched = df.loc[mask, 'title'].tolist()
    if matched:
        print(f"Matched '{keyword}': {matched}")

print(f"\nPrior relevant labels: {(df['label_included'] == 1).sum()}")
print(f"Unlabeled: {df['label_included'].isna().sum()}")

df.to_csv('/content/list_with_priors.csv', index=False)
print("\nSaved")

In [ ]:
# Colab's native port forwarding
import subprocess
import threading
import time

def run_asreview():
    subprocess.Popen(
        ["asreview", "lab"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

t = threading.Thread(target=run_asreview)
t.daemon = True
t.start()

time.sleep(10)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(5001)")
print(f"Open ASReview at:\n{url}")

In [ ]:
import pandas as pd

df = pd.read_csv('/content/asreview_relevant+irrelevant+not_seen_list_with_priors.csv')

relevant = df[df['asreview_label'] == 1]
excluded = df[df['asreview_label'] == 0]
not_seen = df[df['asreview_label'].isna()]

print(f"Relevant (included): {len(relevant)}")
print(f"Excluded:            {len(excluded)}")
print(f"Not yet screened:    {len(not_seen)}")
print(f"Total:               {len(df)}")

In [ ]:
import pandas as pd

df = pd.read_csv('/content/asreview_relevant+irrelevant+not_seen_list_with_priors.csv')
relevant = df[df['asreview_label'] == 1].copy()

relevant[['title', 'author', 'year', 'journal']].reset_index(drop=True).to_csv(
    '/content/fulltext_screening_list.csv', index=False
)

print(f"Papers for full-text review: {len(relevant)}\n")

# Pooled Analysis

In [ ]:
#  Pooled Analysis
#  Model: Random Effects | Estimator: REML
#  ES: Mean Difference (MD)
#  SD Estimation from IQR

import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("/content/Data Extraction.csv", encoding="cp1252")

df = df.rename(columns={
    "author":                                     "study",
    "cal_adeq_intervention_percentage":           "med_i",
    "cal_adeq_intervention_percentage_lower_iqr": "q1_i",
    "cal_adeq_intervention_percentage_upper_iqr": "q3_i",
    "cal_adeq_control_percentage":                "med_c",
    "cal_adeq_control_percentage_lower_iqr":      "q1_c",
    "cal_adeq_control_percentage_upper_iqr":      "q3_c",
    "intervention_n":                             "n_i",
    "control_n":                                  "n_c",
    "protocol_type":                              "protocol",
})


df["protocol"] = df["protocol"].str.replace(r"[^\x00-\x7F]+", "-", regex=True).str.strip()

print("=" * 60)
print("Studies loaded:", df["study"].tolist())
print("Total N:", df["total_N"].sum())
print("k (number of studies):", len(df))
print("=" * 60)

def wan_sd(q1, q3, n):
    """
    Wan et al. (2014) — sample-size corrected SD from IQR.
    SD = IQR / (2 * Φ⁻¹((0.75n − 0.125) / (n + 0.25)))
    Converges to IQR / 1.35 for large n.
    """
    iqr = q3 - q1
    p   = (0.75 * n - 0.125) / (n + 0.25)
    p   = np.clip(p, 1e-6, 1 - 1e-6)
    z   = stats.norm.ppf(p)
    return iqr / (2 * z)

def wan_mean(q1, med, q3):
    """Wan et al. (2014) — mean estimate from Q1, median, Q3."""
    return (q1 + med + q3) / 3

df["mean_i"] = wan_mean(df["q1_i"], df["med_i"], df["q3_i"])
df["mean_c"] = wan_mean(df["q1_c"], df["med_c"], df["q3_c"])
df["sd_i"]   = df.apply(lambda r: wan_sd(r["q1_i"], r["q3_i"], r["n_i"]), axis=1)
df["sd_c"]   = df.apply(lambda r: wan_sd(r["q1_c"], r["q3_c"], r["n_c"]), axis=1)


# PER-STUDY EFFECT SIZES

df["MD"]     = df["mean_i"] - df["mean_c"]
df["var_MD"] = (df["sd_i"]**2 / df["n_i"]) + (df["sd_c"]**2 / df["n_c"])
df["SE_MD"]  = np.sqrt(df["var_MD"])
df["CI_lo"]  = df["MD"] - 1.96 * df["SE_MD"]
df["CI_hi"]  = df["MD"] + 1.96 * df["SE_MD"]

print("\n── STEP 1: Per-Study Effect Sizes ──────────────────────")
print(f"{'Study':<28} {'n(I)':>5} {'n(C)':>5} {'Mean%(I)':>9} {'Mean%(C)':>9} "
      f"{'MD':>7} {'SE':>6} {'95% CI':>20}")
print("-" * 95)
for _, r in df.iterrows():
    ci_str = f"[{r['CI_lo']:+.2f}, {r['CI_hi']:+.2f}]"
    print(f"{r['study']:<28} {r['n_i']:>5} {r['n_c']:>5} "
          f"{r['mean_i']:>9.2f} {r['mean_c']:>9.2f} "
          f"{r['MD']:>+7.2f} {r['SE_MD']:>6.2f} {ci_str:>20}")


# HETEROGENEITY
k   = len(df)
y   = df["MD"].values
v   = df["var_MD"].values
w_f = 1 / v

mu_f = np.average(y, weights=w_f)
Q    = np.sum(w_f * (y - mu_f)**2)
df_Q = k - 1
p_Q  = 1 - stats.chi2.cdf(Q, df=df_Q)

I2 = max(0, (Q - df_Q) / Q * 100)

C        = np.sum(w_f) - np.sum(w_f**2) / np.sum(w_f)
tau2_DL  = max(0, (Q - df_Q) / C)

def reml_tau2(y, v, tau2_init, tol=1e-8, max_iter=1000):
    tau2 = tau2_init if tau2_init > 0 else 0.01
    for _ in range(max_iter):
        w     = 1 / (v + tau2)
        mu    = np.sum(w * y) / np.sum(w)
        score = -0.5 * np.sum(w) + 0.5 * np.sum(w**2 * (y - mu)**2)
        info  =  0.5 * np.sum(w**2)
        tau2_new = max(0, tau2 + score / info)
        if abs(tau2_new - tau2) < tol:
            return tau2_new
        tau2 = tau2_new
    return tau2

tau2_REML = reml_tau2(y, v, tau2_DL)
tau_REML  = np.sqrt(tau2_REML)

H2   = Q / df_Q
lnH  = np.log(np.sqrt(H2)) if H2 > 1 else 0
se_lnH = 0.5 * np.sqrt(
    (1/(2*(df_Q - 1))) * (1 - 1/(8*(df_Q - 1)))**(-3)
) if df_Q > 1 else 0
lnH_lo = lnH - 1.96 * se_lnH
lnH_hi = lnH + 1.96 * se_lnH
H_lo   = np.exp(lnH_lo)
H_hi   = np.exp(lnH_hi)
I2_lo  = max(0, (H_lo**2 - 1) / H_lo**2 * 100)
I2_hi  = min(100, (H_hi**2 - 1) / H_hi**2 * 100)

print("\n\n── STEP 2: Heterogeneity ────────────────────────────────")
print(f"  Cochran's Q  = {Q:.3f}  (df = {df_Q},  p = {p_Q:.4f})")
print(f"  I²           = {I2:.1f}%  (95% CI: {I2_lo:.1f}% – {I2_hi:.1f}%)")
print(f"  τ²  (REML)   = {tau2_REML:.4f}")
print(f"  τ   (REML)   = {tau_REML:.4f}  [in same units as MD, i.e. %]")
print(f"  τ²  (DL)     = {tau2_DL:.4f}  [DerSimonian-Laird, for reference]")

# Heterogeneity labels
if I2 < 25:
    het_label = "Low heterogeneity"
elif I2 < 50:
    het_label = "Moderate heterogeneity"
elif I2 < 75:
    het_label = "Substantial heterogeneity"
else:
    het_label = "Considerable heterogeneity"
print(f"\n  ► {het_label} (I² = {I2:.1f}%)")


# POOLED RANDOM-EFFECTS ESTIMATE
w_re     = 1 / (v + tau2_REML)
mu_re    = np.sum(w_re * y) / np.sum(w_re)
se_re    = np.sqrt(1 / np.sum(w_re))
ci_lo_re = mu_re - 1.96 * se_re
ci_hi_re = mu_re + 1.96 * se_re
z_re     = mu_re / se_re
p_re     = 2 * (1 - stats.norm.cdf(abs(z_re)))

pi_lo = mu_re - 1.96 * np.sqrt(tau2_REML + se_re**2)
pi_hi = mu_re + 1.96 * np.sqrt(tau2_REML + se_re**2)

df["w_re_pct"] = w_re / np.sum(w_re) * 100

print("\n\n── STEP 3: Pooled Random-Effects Estimate (REML) ───────")
print(f"  Pooled MD    = {mu_re:+.2f}%")
print(f"  95% CI       = [{ci_lo_re:.2f}, {ci_hi_re:.2f}]")
print(f"  SE           = {se_re:.4f}")
print(f"  Z            = {z_re:.3f}")
print(f"  p-value      = {p_re:.4f}")
print(f"  Prediction Interval = [{pi_lo:.2f}, {pi_hi:.2f}]")
print(f"  (The PI tells you where 95% of true effects would lie")
print(f"   in a new similar study — important given high heterogeneity)")

print("\n── Per-Study RE Weights ─────────────────────────────────")
print(f"{'Study':<28} {'Weight (%)':>10}")
print("-" * 40)
for _, r in df.iterrows():
    print(f"{r['study']:<28} {r['w_re_pct']:>10.1f}%")

print("\n" + "=" * 60)
print("  SECTION 4 — POOLED ANALYSIS COMPLETE")
print("=" * 60)
print(f"""
  Pooled MD (RE, REML)   = {mu_re:+.2f}%
  95% CI                 = [{ci_lo_re:.2f}, {ci_hi_re:.2f}]
  p-value                = {p_re:.4f}
  Prediction Interval    = [{pi_lo:.2f}, {pi_hi:.2f}]
  I²                     = {I2:.1f}%  (95% CI: {I2_lo:.1f}–{I2_hi:.1f}%)
  τ (REML)               = {tau_REML:.2f}%
  Cochran Q              = {Q:.2f}  (p = {p_Q:.4f})
""")

In [ ]:
# Pooled forest plot
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

COL_NURSE  = "#C0392B"   # Nurse-led protocol      ─ crimson
COL_NONNU  = "#1B6CA8"   # Non-nurse-led           ─ steel blue
COL_POOL   = "#2C2C2C"   # Pooled diamond          ─ near black
COL_GRID   = "#DADADA"   # Gridlines
COL_HEADER = "#1a1a2e"   # Header text
COL_BODY   = "#2d2d2d"   # Body text
COL_SUB    = "#555555"   # Subtext / CI

PROTOCOL_COLOR = {
    "Nurse-led protocol": COL_NURSE,
    "Non-nurse-led":      COL_NONNU,
}

fig = plt.figure(figsize=(14, 7))
fig.patch.set_facecolor("#FAFBFC")

ax_text = fig.add_axes([0, 0, 1, 1])
ax_text.set_xlim(0, 1)
ax_text.set_ylim(0, 1)
ax_text.axis("off")
ax_text.set_facecolor("#FAFBFC")

PLOT_LEFT   = 0.345
PLOT_WIDTH  = 0.24
PLOT_BOTTOM = 0.11
PLOT_HEIGHT = 0.76
ax = fig.add_axes([PLOT_LEFT, PLOT_BOTTOM, PLOT_WIDTH, PLOT_HEIGHT])
ax.set_facecolor("#FAFBFC")

n = len(df)
y_positions = np.arange(n, 0, -1, dtype=float)
POOL_Y      = 0.0

x_min, x_max = -40, 60
ax.set_xlim(x_min, x_max)
ax.set_ylim(POOL_Y - 1.2, n + 2.2)

for xg in [-20, 0, 20, 40, 60]:
    if xg == 0:
        ax.axvline(xg, ymin=0.15, ymax=0.90,
                   color="black", linewidth=1.6, zorder=1)
    else:
        ax.axvline(xg, color=COL_GRID, linewidth=0.7,
                   alpha=0.4, zorder=0)

X_STUDY = 0.010
X_PROTO = 0.120
X_COMP  = 0.245
X_MD    = 0.640
X_WT    = 0.730

HEADER_Y_fig = 0.905
SEP_Y_fig    = 0.898

def data_to_fig_y(data_y):
    ax_y = (data_y - (POOL_Y - 1.2)) / (n + 2.2 - (POOL_Y - 1.2))
    return PLOT_BOTTOM + ax_y * PLOT_HEIGHT

header_kwargs = dict(transform=ax_text.transAxes, va="center",
                     fontsize=9.2, fontweight="bold",
                     color=COL_HEADER, fontfamily="DejaVu Sans")

ax_text.text(X_STUDY, HEADER_Y_fig, "Study ID",    ha="left",   **header_kwargs)
ax_text.text(X_PROTO, HEADER_Y_fig, "Protocol",    ha="left",   **header_kwargs)
ax_text.text(X_COMP,  HEADER_Y_fig, "Comparator",  ha="left",   **header_kwargs)
ax_text.text(X_MD,    HEADER_Y_fig, "MD (95% CI)", ha="center", **header_kwargs)
ax_text.text(X_WT,    HEADER_Y_fig, "Weight (%)",  ha="center", **header_kwargs)

ax.text((x_min + 0) / 2, n + 1.55,
        "◄ Favors Control", ha="center", va="center",
        fontsize=7.8, color="#8B0000", style="italic",
        fontfamily="DejaVu Sans")
ax.text((0 + x_max) / 2, n + 1.55,
        "Favors Intervention ►", ha="center", va="center",
        fontsize=7.8, color="#145A32", style="italic",
        fontfamily="DejaVu Sans")
ax.text(0, n + 2.0,
        "Caloric Adequacy — Mean Difference (%)",
        ha="center", va="center", fontsize=8.5,
        fontweight="bold", color=COL_HEADER,
        fontfamily="DejaVu Sans")

ax_text.axhline(SEP_Y_fig, color="#AAAAAA", linewidth=0.1,
                xmin=0.01, xmax=0.95)

for i, (_, row) in enumerate(df.iterrows()):
    yp    = y_positions[i]
    proto = row["protocol"]
    color = PROTOCOL_COLOR.get(proto, "#555555")

    ci_lo_plot = max(row["CI_lo"], x_min + 0.5)
    ci_hi_plot = min(row["CI_hi"], x_max - 0.5)
    ax.plot([ci_lo_plot, ci_hi_plot], [yp, yp],
            color=color, linewidth=1.8,
            solid_capstyle="round", zorder=2)

    cap_h = 0.16
    for cx in [ci_lo_plot, ci_hi_plot]:
        ax.plot([cx, cx], [yp - cap_h, yp + cap_h],
                color=color, linewidth=1.8, zorder=2)

    sq = 55 + row["w_re_pct"] * 16
    ax.scatter(row["MD"], yp, marker="s", s=sq,
               color=color, zorder=3,
               edgecolors="white", linewidths=0.5)

    fy = data_to_fig_y(yp)
    text_kw = dict(transform=ax_text.transAxes, va="center",
                   fontsize=8.4, color=COL_BODY,
                   fontfamily="DejaVu Sans")

    ax_text.text(X_STUDY, fy, row["study"],
                 ha="left", fontweight="semibold", **text_kw)

    ax_text.text(X_PROTO, fy, proto,
                 ha="left", color=color, fontsize=8.0,
                 transform=ax_text.transAxes, va="center",
                 fontfamily="DejaVu Sans")

    ax_text.text(X_COMP, fy, row["comparator"],
                 ha="left", **text_kw)

    md_str = f"{row['MD']:.1f}  [{row['CI_lo']:.1f}, {row['CI_hi']:.1f}]"
    ax_text.text(X_MD, fy, md_str, ha="center",
                 fontsize=9, color=COL_SUB,
                 transform=ax_text.transAxes, va="center",
                 fontfamily="DejaVu Sans Mono")

    ax_text.text(X_WT, fy, f"{row['w_re_pct']:.1f}%",
                 ha="center", fontsize=8.4, color=COL_BODY,
                 transform=ax_text.transAxes, va="center",
                 fontfamily="DejaVu Sans")

    if i % 2 == 0:
        ax.axhspan(yp - 0.45, yp + 0.45,
                   facecolor="#F0F4F8", alpha=0.45, zorder=0)

ax.axhline(0.55, color="#888888", linewidth=0.9,
           linestyle="-", xmin=0.0, xmax=1.0, zorder=3)

dh = 0.40
dx = [ci_lo_re, mu_re,     ci_hi_re, mu_re,     ci_lo_re]
dy = [POOL_Y,   POOL_Y+dh, POOL_Y,   POOL_Y-dh, POOL_Y]
ax.fill(dx, dy, color=COL_POOL, alpha=0.88, zorder=4)
ax.plot(dx, dy, color="#000000", linewidth=0.9, zorder=5)

fy_pool = data_to_fig_y(POOL_Y)
pool_kw = dict(transform=ax_text.transAxes, va="center",
               fontsize=8.8, fontweight="bold",
               color=COL_POOL, fontfamily="DejaVu Sans")

ax_text.text(X_STUDY, fy_pool, "Pooled Estimate",    ha="left", **pool_kw)
ax_text.text(X_PROTO, fy_pool, "RE Model (REML)",    ha="left", **pool_kw)
ax_text.text(X_COMP,  fy_pool, "Mixed Comparators",  ha="left", **pool_kw)

pool_md_str = f"{mu_re:.1f}  [{ci_lo_re:.1f}, {ci_hi_re:.1f}]"
ax_text.text(X_MD, fy_pool, pool_md_str, ha="center",
             fontsize=8.8, fontweight="bold", color=COL_POOL,
             transform=ax_text.transAxes, va="center",
             fontfamily="DejaVu Sans Mono")
ax_text.text(X_WT, fy_pool, "100.0%", ha="center",
             fontsize=8.8, fontweight="bold", color=COL_POOL,
             transform=ax_text.transAxes, va="center",
             fontfamily="DejaVu Sans")

ax.set_yticks([])
ax.set_xticks([-20, 0, 20, 40, 60])
ax.set_xticklabels(["-20", "0", "20", "40", "60"],
                   fontsize=8.5, color=COL_BODY)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_linewidth(0.8)
ax.spines["bottom"].set_color("#AAAAAA")
ax.tick_params(axis="x", length=3, color="#AAAAAA")

legend_handles = [
    mpatches.Patch(color=COL_NURSE, label="Nurse-led protocol"),
    mpatches.Patch(color=COL_NONNU, label="Non-nurse-led"),
    mpatches.Patch(color=COL_POOL,  label="Pooled Effect"),
]
ax_text.legend(
    handles=legend_handles,
    loc="upper left",
    bbox_to_anchor=(X_STUDY, 0.12),
    fontsize=9,
    frameon=True,
    ncol=2,
)

plt.savefig("forest_plot_caloric_adequacy.png",
            dpi=600, bbox_inches="tight",
            pad_inches=0, facecolor="#FAFBFC")
plt.show()

# Publication bias

In [ ]:
# FUNNEL PLOT
fig2, ax2 = plt.subplots(figsize=(8, 6.5))
fig2.patch.set_facecolor("white")
ax2.set_facecolor("white")

se_vals  = df["SE_MD"].values
md_vals  = df["MD"].values
max_se   = se_vals.max() * 1.15

se_range = np.linspace(0, max_se, 200)
ax2.fill_betweenx(se_range,
                  mu_re - 1.96 * se_range,
                  mu_re + 1.96 * se_range,
                  alpha=0.10, color="#2C7BB6", label="95% pseudo-CI")
ax2.plot(mu_re - 1.96 * se_range, se_range,
         color="#2C7BB6", linewidth=1.2, linestyle="--")
ax2.plot(mu_re + 1.96 * se_range, se_range,
         color="#2C7BB6", linewidth=1.2, linestyle="--")

ax2.axvline(mu_re, color="#333333", linewidth=1.2,
            linestyle="-", label=f"Pooled MD = {mu_re:.2f}%")
ax2.axvline(0, color="black", linewidth=0.7,
            linestyle=":", alpha=0.5, label="No effect (MD = 0)")

for _, row in df.iterrows():
    color = PROTOCOL_COLOR.get(row["protocol"], "#555555")
    ax2.scatter(row["MD"], row["SE_MD"],
                color=color, s=70, zorder=3,
                edgecolors="white", linewidths=0.5)
    ax2.annotate(row["study"].split()[0],
                 (row["MD"], row["SE_MD"]),
                 textcoords="offset points", xytext=(5, 3),
                 fontsize=7.5, color="#333333")

ax2.set_xlabel("Mean Difference in Caloric Adequacy (%)", fontsize=10)
ax2.set_ylabel("Standard Error (SE)", fontsize=10)
ax2.invert_yaxis()

import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

legend_handles = [
    mpatches.Patch(color=COL_NURSE, label="Nurse-led protocol"),
    mpatches.Patch(color=COL_NONNU, label="Non-nurse-led"),
    Line2D([0], [0], color="#333333", linewidth=1.2,
           linestyle="-",  label=f"Pooled MD = {mu_re:.2f}%"),
    Line2D([0], [0], color="#2C7BB6", linewidth=1.2,
           linestyle="--", label="95% pseudo-CI"),
    Line2D([0], [0], color="black",   linewidth=0.7,
           linestyle=":",  label="No effect (MD = 0)"),
]
ax2.legend(handles=legend_handles, fontsize=8.2,
           frameon=True, loc="upper right")

plt.tight_layout()
plt.savefig("funnel_plot_caloric_adequacy.png",
            dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# EGGER'S TEST
import numpy as np
import pandas as pd
from scipy import stats

df_clean = df.dropna(subset=["MD", "SE_MD"]).copy()
df_clean = df_clean[df_clean["SE_MD"] > 0]

y  = df_clean["MD"].values
se = df_clean["SE_MD"].values
v  = se**2
k  = len(y)

precision  = 1 / se
std_effect = y / se

X        = np.vstack([np.ones_like(precision), precision]).T
beta_hat = np.linalg.inv(X.T @ X) @ (X.T @ std_effect)
intercept, slope = beta_hat

residuals     = std_effect - X @ beta_hat
sigma2        = np.sum(residuals**2) / (k - 2)
cov_beta      = sigma2 * np.linalg.inv(X.T @ X)
intercept_se  = np.sqrt(cov_beta[0, 0])
slope_se      = np.sqrt(cov_beta[1, 1])

t_stat = intercept / intercept_se
p_val  = 2 * (1 - stats.t.cdf(abs(t_stat), df=k - 2))

print("\n── Egger's Test (Publication Bias) ─────────────────────")
print(f"  Intercept   = {intercept:.4f}  (SE = {intercept_se:.4f})")
print(f"  Slope       = {slope:.4f}  (SE = {slope_se:.4f})")
print(f"  t-statistic = {t_stat:.4f}  (df = {k - 2})")
print(f"  p-value     = {p_val:.4f}")
if k < 10:
    print("  ⚠ n < 10 → very low power; interpret cautiously")
if p_val < 0.05:
    print("  ⚠ Significant asymmetry detected (p < 0.05)")
else:
    print("  ✓ No significant asymmetry (p ≥ 0.05)")

def trim_and_fill(y, v, tau2, max_iter=100):
    """
    Duval & Tweedie (2000) trim-and-fill.
    Detects right-side asymmetry; mirrors missing studies on the left.
    Returns: k0, adjusted pooled MD, lower CI, upper CI
    """
    y = np.asarray(y, dtype=float)
    v = np.asarray(v, dtype=float)

    w  = 1 / (v + tau2)
    mu = np.sum(w * y) / np.sum(w)

    k0 = 0
    for _ in range(max_iter):
        dev     = y - mu
        k_right = np.sum(dev > 0)
        k_left  = np.sum(dev < 0)
        k0_new  = max(0, k_right - k_left)

        if k0_new == 0:
            k0 = 0
            break

        idx    = np.argsort(dev)[::-1][:k0_new]
        y_trim = np.delete(y, idx)
        v_trim = np.delete(v, idx)
        w_trim = 1 / (v_trim + tau2)
        mu_new = np.sum(w_trim * y_trim) / np.sum(w_trim)

        if np.isclose(mu, mu_new, atol=1e-6):
            k0 = k0_new
            break

        mu = mu_new
        k0 = k0_new

    if k0 > 0:
        idx_top  = np.argsort(y - mu)[::-1][:k0]
        y_mirror = 2 * mu - y[idx_top]
        v_mirror = v[idx_top]
        y_aug    = np.concatenate([y, y_mirror])
        v_aug    = np.concatenate([v, v_mirror])
    else:
        y_aug, v_aug = y, v

    w_aug  = 1 / (v_aug + tau2)
    mu_adj = np.sum(w_aug * y_aug) / np.sum(w_aug)
    se_adj = np.sqrt(1 / np.sum(w_aug))

    return k0, mu_adj, mu_adj - 1.96 * se_adj, mu_adj + 1.96 * se_adj

k0, mu_tf, ci_lo_tf, ci_hi_tf = trim_and_fill(y, v, tau2_REML)

print("\n── Trim-and-Fill (Duval & Tweedie) ─────────────────────")
print(f"  Estimated missing studies (k0) = {k0}")
print(f"  Adjusted pooled MD  = {mu_tf:.2f}%")
print(f"  Adjusted 95% CI     = [{ci_lo_tf:.2f}, {ci_hi_tf:.2f}]")
print(f"  Original pooled MD  = {mu_re:.2f}%  [{ci_lo_re:.2f}, {ci_hi_re:.2f}]")
if k0 == 0:
    print("  ✓ No studies trimmed — no adjustment needed.")
else:
    print(f"  ⚠ {k0} study/studies imputed — compare adjusted vs original above")

# Subgroup analysis

In [ ]:
#  SUBGROUP ANALYSIS
#  Model: Random Effects (REML) | Effect: Mean Difference (MD)

import numpy as np
import pandas as pd
from scipy import stats

def pool_reml(y, v, label=""):
    """
    Pool a subgroup with its own REML tau².
    Returns dict of all key statistics.
    """
    k = len(y)
    if k == 0:
        return None
    if k == 1:
        mu  = y[0]
        se  = np.sqrt(v[0])
        return dict(k=1, mu=mu, se=se,
                    ci_lo=mu - 1.96*se, ci_hi=mu + 1.96*se,
                    tau2=0.0, tau=0.0,
                    Q=0.0, df_Q=0, p_Q=np.nan,
                    I2=0.0, label=label)

    w_f  = 1 / v
    mu_f = np.average(y, weights=w_f)
    Q    = np.sum(w_f * (y - mu_f)**2)
    df_Q = k - 1
    p_Q  = 1 - stats.chi2.cdf(Q, df=df_Q)
    I2   = max(0, (Q - df_Q) / Q * 100)

    C       = np.sum(w_f) - np.sum(w_f**2) / np.sum(w_f)
    tau2_DL = max(0, (Q - df_Q) / C)
    tau2    = reml_tau2(y, v, tau2_DL)
    tau     = np.sqrt(tau2)

    w_re = 1 / (v + tau2)
    mu   = np.sum(w_re * y) / np.sum(w_re)
    se   = np.sqrt(1 / np.sum(w_re))

    return dict(k=k, mu=mu, se=se,
                ci_lo=mu - 1.96*se, ci_hi=mu + 1.96*se,
                tau2=tau2, tau=tau,
                Q=Q, df_Q=df_Q, p_Q=p_Q,
                I2=I2, label=label)


def print_subgroup(res):
    if res is None:
        print("  (no studies)")
        return
    print(f"  k            = {res['k']}")
    print(f"  Pooled MD    = {res['mu']:+.2f}%")
    print(f"  95% CI       = [{res['ci_lo']:.2f}, {res['ci_hi']:.2f}]")
    print(f"  I²           = {res['I2']:.1f}%")
    print(f"  τ (REML)     = {res['tau']:.2f}%")
    if res['k'] > 1:
        print(f"  Cochran Q    = {res['Q']:.3f}  (df={res['df_Q']}, p={res['p_Q']:.4f})")


def between_group_test(res_list):
    """
    Wald-type test for subgroup difference.
    χ²(df = n_groups-1) = Σ [ (mu_j - mu_pool)² / se_j² ]
    """
    mus  = np.array([r["mu"]  for r in res_list])
    ses  = np.array([r["se"]  for r in res_list])
    w    = 1 / ses**2
    mu_p = np.average(mus, weights=w)
    Q_b  = np.sum(w * (mus - mu_p)**2)
    df_b = len(res_list) - 1
    p_b  = 1 - stats.chi2.cdf(Q_b, df=df_b)
    return Q_b, df_b, p_b


y_all = df["MD"].values
v_all = df["var_MD"].values

#  SUBGROUP 1 — Protocol Type
#  Nurse-led protocol  vs  Non-nurse-led
print("\n" + "="*60)
print("  SUBGROUP 1: Protocol Type")
print("="*60)

for grp_label, grp_val in [("Nurse-led protocol", "Nurse-led protocol"),
                             ("Non-nurse-led",      "Non-nurse-led")]:
    mask = df["protocol"] == grp_val
    y_g  = df.loc[mask, "MD"].values
    v_g  = df.loc[mask, "var_MD"].values
    res  = pool_reml(y_g, v_g, label=grp_label)
    print(f"\n  ── {grp_label} ──")
    print_subgroup(res)

res_nl  = pool_reml(df.loc[df["protocol"] == "Nurse-led protocol", "MD"].values,
                    df.loc[df["protocol"] == "Nurse-led protocol", "var_MD"].values)
res_nml = pool_reml(df.loc[df["protocol"] == "Non-nurse-led",      "MD"].values,
                    df.loc[df["protocol"] == "Non-nurse-led",       "var_MD"].values)
Q_b, df_b, p_b = between_group_test([res_nl, res_nml])
print(f"\n  ── Between-group test ──")
print(f"  Q_between = {Q_b:.3f}  (df = {df_b},  p = {p_b:.4f})")
if p_b < 0.05:
    print("  ► Significant subgroup difference (p < 0.05)")
else:
    print("  ► No significant subgroup difference (p ≥ 0.05)")



#  SUBGROUP 2 — Comparator
#  Standard care  vs  Historical Control
print("\n" + "="*60)
print("  SUBGROUP 2: Comparator")
print("="*60)

for grp_label, grp_val in [("Standard care",     "Standard care"),
                             ("Historical Control","Historical Control")]:
    mask = df["comparator"] == grp_val
    y_g  = df.loc[mask, "MD"].values
    v_g  = df.loc[mask, "var_MD"].values
    res  = pool_reml(y_g, v_g, label=grp_label)
    print(f"\n  ── {grp_label} ──")
    print_subgroup(res)

res_sc  = pool_reml(df.loc[df["comparator"] == "Standard care",      "MD"].values,
                    df.loc[df["comparator"] == "Standard care",       "var_MD"].values)
res_hc  = pool_reml(df.loc[df["comparator"] == "Historical Control", "MD"].values,
                    df.loc[df["comparator"] == "Historical Control",  "var_MD"].values)
Q_b, df_b, p_b = between_group_test([res_sc, res_hc])
print(f"\n  ── Between-group test ──")
print(f"  Q_between = {Q_b:.3f}  (df = {df_b},  p = {p_b:.4f})")
if p_b < 0.05:
    print("  ► Significant subgroup difference (p < 0.05)")
else:
    print("  ► No significant subgroup difference (p ≥ 0.05)")

#  SUBGROUP 3 — Study Design
#  Observational  vs  Interventional

print("\n" + "="*60)
print("  SUBGROUP 3: Study Design")
print("="*60)

for grp_label, grp_val in [("Observational",  "Observational"),
                             ("Interventional", "Interventional")]:
    mask = df["study_design"] == grp_val
    y_g  = df.loc[mask, "MD"].values
    v_g  = df.loc[mask, "var_MD"].values
    res  = pool_reml(y_g, v_g, label=grp_label)
    print(f"\n  ── {grp_label} ──")
    print_subgroup(res)

res_obs = pool_reml(df.loc[df["study_design"] == "Observational",  "MD"].values,
                    df.loc[df["study_design"] == "Observational",  "var_MD"].values)
res_int = pool_reml(df.loc[df["study_design"] == "Interventional", "MD"].values,
                    df.loc[df["study_design"] == "Interventional", "var_MD"].values)
Q_b, df_b, p_b = between_group_test([res_obs, res_int])
print(f"\n  ── Between-group test ──")
print(f"  Q_between = {Q_b:.3f}  (df = {df_b},  p = {p_b:.4f})")
if p_b < 0.05:
    print("  ► Significant subgroup difference (p < 0.05)")
else:
    print("  ► No significant subgroup difference (p ≥ 0.05)")

print("="*60)
print("  SUBGROUP ANALYSIS COMPLETE")
print("="*60)

In [ ]:
# Subgroup forest plot
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

COL_NURSE  = "#C0392B"
COL_NONNU  = "#1B6CA8"
COL_GRID   = "#DADADA"
COL_HEADER = "#1a1a2e"
COL_BODY   = "#2d2d2d"
COL_SUB    = "#555555"

PROTOCOL_COLOR = {
    "Nurse-led protocol": COL_NURSE,
    "Non-nurse-led":      COL_NONNU,
}

SUBGROUPS = [
    {
        "header": "Protocol Type",
        "strata": [
            ("Nurse-led protocol", "protocol",     "Nurse-led protocol", "#C0392B"),
            ("Non-nurse-led",      "protocol",     "Non-nurse-led",      "#1B6CA8"),
        ],
    },
    {
        "header": "Comparator",
        "strata": [
            ("Standard care",      "comparator",   "Standard care",      "#8E44AD"),
            ("Historical Control", "comparator",   "Historical Control", "#D35400"),
        ],
    },
    {
        "header": "Study Design",
        "strata": [
            ("Observational",      "study_design", "Observational",      "#16A085"),
            ("Interventional",     "study_design", "Interventional",     "#B7950B"),
        ],
    },
]

rows = []

for sg in SUBGROUPS:
    rows.append({"type": "section_header", "label": sg["header"]})
    for stratum_label, col, val, dia_col in sg["strata"]:
        mask   = df[col] == val
        sub_df = df[mask].copy()
        y_g    = sub_df["MD"].values
        v_g    = sub_df["var_MD"].values
        res    = pool_reml(y_g, v_g, label=stratum_label)

        for _, row in sub_df.iterrows():
            rows.append({
                "type":       "study",
                "study":      row["study"],
                "protocol":   row["protocol"],
                "comparator": row["comparator"],
                "MD":         row["MD"],
                "CI_lo":      row["CI_lo"],
                "CI_hi":      row["CI_hi"],
                "w_re_pct":   row["w_re_pct"],
            })

        rows.append({
            "type":  "diamond",
            "label": f"Subtotal: {stratum_label}  (k={res['k']})",
            "mu":    res["mu"],
            "ci_lo": res["ci_lo"],
            "ci_hi": res["ci_hi"],
            "I2":    res["I2"],
            "tau":   res["tau"],
            "color": dia_col,
            "k":     res["k"],
        })

    rows.append({"type": "spacer"})

STUDY_H   = 1.0
HEADER_H  = 1.6
DIAMOND_H = 1.2
SPACER_H  = 0.8

y_cursor = 0.0
for r in rows:
    if   r["type"] == "section_header": h = HEADER_H
    elif r["type"] == "study":          h = STUDY_H
    elif r["type"] == "diamond":        h = DIAMOND_H
    else:                               h = SPACER_H
    r["y"] = y_cursor
    y_cursor += h

total_height = y_cursor
for r in rows:
    r["yp"] = total_height - r["y"]

fig_height = max(16, total_height * 0.52)
fig = plt.figure(figsize=(20, fig_height))
fig.patch.set_facecolor("#FAFBFC")

ax_text = fig.add_axes([0, 0, 1, 1])
ax_text.set_xlim(0, 1)
ax_text.set_ylim(0, 1)
ax_text.axis("off")
ax_text.set_facecolor("#FAFBFC")

PLOT_LEFT   = 0.30
PLOT_WIDTH  = 0.25
PLOT_BOTTOM = 0.04
PLOT_HEIGHT = 0.92

ax = fig.add_axes([PLOT_LEFT, PLOT_BOTTOM, PLOT_WIDTH, PLOT_HEIGHT])
ax.set_facecolor("#FAFBFC")

y_min = 0
y_max = total_height + HEADER_H * 0.45
ax.set_ylim(y_min, y_max)
x_min, x_max = -60, 80
ax.set_xlim(x_min, x_max)

def data_to_fig_frac(yp):
    """data-y → figure fraction (for ax_text.transAxes)"""
    ax_frac = (yp - y_min) / (y_max - y_min)
    return PLOT_BOTTOM + ax_frac * PLOT_HEIGHT

for xg in [-40, -20, 0, 20, 40, 60, 80]:
    if xg == 0:
        ax.axvline(xg, color="black", linewidth=1.5, zorder=1)
    else:
        ax.axvline(xg, color=COL_GRID, linewidth=0.6, alpha=0.5, zorder=0)

X_STUDY = 0.005
X_PROTO = 0.155
X_COMP  = 0.230
X_MD    = 0.600
X_WT    = 0.670

HEADER_FIG_Y = 0.978
hdr_kw = dict(transform=ax_text.transAxes, va="center",
              fontsize=11.5, fontweight="bold",
              color=COL_HEADER, fontfamily="DejaVu Sans")

ax_text.text(X_STUDY, HEADER_FIG_Y, "Study ID",    ha="left",   **hdr_kw)
ax_text.text(X_PROTO, HEADER_FIG_Y, "Protocol",    ha="left",   **hdr_kw)
ax_text.text(X_COMP,  HEADER_FIG_Y, "Comparator",  ha="left",   **hdr_kw)
ax_text.text(X_MD,    HEADER_FIG_Y, "MD (95% CI)", ha="center", **hdr_kw)
ax_text.text(X_WT,    HEADER_FIG_Y, "Weight (%)",  ha="center", **hdr_kw)

ax_text.axhline(HEADER_FIG_Y - 0.012, color="#AAAAAA", linewidth=0.6,
                xmin=0.005, xmax=0.99)

study_count = 0

for r in rows:
    yp     = r["yp"]
    fy_fig = data_to_fig_frac(yp)


    if r["type"] == "section_header":
        ax_text.text(X_STUDY, fy_fig, r["label"],
                     transform=ax_text.transAxes,
                     ha="left", va="center",
                     fontsize=12, fontweight="bold",
                     color=COL_HEADER, fontfamily="DejaVu Sans")
        ax_text.axhline(fy_fig - 0.010, color=COL_HEADER,
                        linewidth=0.6, xmin=X_STUDY, xmax=0.99)
        ax.axhspan(yp - HEADER_H * 0.45, yp + HEADER_H * 0.45,
                   facecolor="#E8EDF2", alpha=0.55, zorder=0)

        ax.text((x_min + 0) / 2, yp + HEADER_H * 0.05,
                "◄ Favors Control", ha="center", va="center",
                fontsize=10, color="#8B0000", style="italic",
                fontfamily="DejaVu Sans")
        ax.text((0 + x_max) / 2, yp + HEADER_H * 0.05,
                "Favors Intervention ►", ha="center", va="center",
                fontsize=10, color="#145A32", style="italic",
                fontfamily="DejaVu Sans")
        continue

    if r["type"] == "spacer":
        continue

    if r["type"] == "study":
        color     = PROTOCOL_COLOR.get(r["protocol"], "#555555")
        ci_lo_plt = max(r["CI_lo"], x_min + 0.5)
        ci_hi_plt = min(r["CI_hi"], x_max - 0.5)

        ax.plot([ci_lo_plt, ci_hi_plt], [yp, yp],
                color=color, linewidth=1.6,
                solid_capstyle="round", zorder=2)
        cap_h = 0.13
        for cx in [ci_lo_plt, ci_hi_plt]:
            ax.plot([cx, cx], [yp - cap_h, yp + cap_h],
                    color=color, linewidth=1.6, zorder=2)

        sq = 38 + r["w_re_pct"] * 13
        ax.scatter(r["MD"], yp, marker="s", s=sq,
                   color=color, zorder=3,
                   edgecolors="white", linewidths=0.4)

        if study_count % 2 == 0:
            ax.axhspan(yp - 0.44, yp + 0.44,
                       facecolor="#F0F4F8", alpha=0.38, zorder=0)
        study_count += 1

        text_kw = dict(transform=ax_text.transAxes, va="center",
                       fontsize=10.2, color=COL_BODY,
                       fontfamily="DejaVu Sans")

        ax_text.text(X_STUDY, fy_fig, r["study"],
                     ha="left", fontweight="semibold", **text_kw)
        ax_text.text(X_PROTO, fy_fig, r["protocol"],
                     ha="left", color=color, fontsize=9.8,
                     transform=ax_text.transAxes, va="center",
                     fontfamily="DejaVu Sans")
        ax_text.text(X_COMP, fy_fig, r["comparator"],
                     ha="left", **text_kw)
        md_str = f"{r['MD']:.1f}  [{r['CI_lo']:.1f}, {r['CI_hi']:.1f}]"
        ax_text.text(X_MD, fy_fig, md_str, ha="center",
                     fontsize=10.2, color=COL_SUB,
                     transform=ax_text.transAxes, va="center",
                     fontfamily="DejaVu Sans Mono")
        ax_text.text(X_WT, fy_fig, f"{r['w_re_pct']:.1f}%",
                     ha="center", fontsize=10.2, color=COL_BODY,
                     transform=ax_text.transAxes, va="center",
                     fontfamily="DejaVu Sans")

    elif r["type"] == "diamond":
        dia_col   = r["color"]
        ci_lo_plt = max(r["ci_lo"], x_min + 0.5)
        ci_hi_plt = min(r["ci_hi"], x_max - 0.5)

        ax.axhline(yp + 0.55, color="#888888", linewidth=0.75,
                   linestyle="-", xmin=0, xmax=1, zorder=3)

        dh = 0.36
        dx = [ci_lo_plt, r["mu"],     ci_hi_plt, r["mu"],     ci_lo_plt]
        dy = [yp,        yp + dh,     yp,        yp - dh,     yp]
        ax.fill(dx, dy, color=dia_col, alpha=0.85, zorder=4)
        ax.plot(dx, dy, color="black", linewidth=0.7, zorder=5)

        pool_kw = dict(transform=ax_text.transAxes, va="center",
                       fontsize=10.3, fontweight="bold",
                       color=dia_col, fontfamily="DejaVu Sans")

        ax_text.text(X_STUDY, fy_fig, r["label"],
                     ha="left", **pool_kw)
        ax_text.text(X_PROTO, fy_fig,
                     f"I²={r['I2']:.0f}%  τ={r['tau']:.1f}%",
                     ha="left", fontsize=9.5, color=dia_col,
                     transform=ax_text.transAxes, va="center",
                     fontfamily="DejaVu Sans")
        md_str = f"{r['mu']:.1f}  [{r['ci_lo']:.1f}, {r['ci_hi']:.1f}]"
        ax_text.text(X_MD, fy_fig, md_str, ha="center",
                     fontsize=10.3, fontweight="bold", color=dia_col,
                     transform=ax_text.transAxes, va="center",
                     fontfamily="DejaVu Sans Mono")
        ax_text.text(X_WT, fy_fig, "—",
                     ha="center", fontsize=10.3, color=dia_col,
                     transform=ax_text.transAxes, va="center",
                     fontfamily="DejaVu Sans")

ax.set_yticks([])
ax.set_xticks([-40, -20, 0, 20, 40, 60, 80])
ax.set_xticklabels(["-40", "-20", "0", "20", "40", "60", "80"],
                   fontsize=10.5, color=COL_BODY)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_linewidth(0.8)
ax.spines["bottom"].set_color("#AAAAAA")
ax.tick_params(axis="x", length=3, color="#AAAAAA")
ax.set_xlabel("Mean Difference in Caloric Adequacy (%)",
              fontsize=11, color=COL_BODY, labelpad=6)

legend_handles = [
    # Protocol colours (study markers)
    mpatches.Patch(color="#C0392B", label="Nurse-led protocol"),
    mpatches.Patch(color="#1B6CA8", label="Non-nurse-led"),
    # Subgroup diamond colours
    mpatches.Patch(color="#8E44AD", label="Standard care"),
    mpatches.Patch(color="#D35400", label="Historical Control"),
    mpatches.Patch(color="#16A085", label="Observational"),
    mpatches.Patch(color="#B7950B", label="Interventional"),
]
ax_text.legend(
    handles=legend_handles,
    loc="lower left",
    bbox_to_anchor=(X_STUDY, 0.01),
    fontsize=10,
    frameon=True,
    ncol=2,
)

plt.savefig("subgroup_forest_plot.png",
            dpi=600, bbox_inches="tight",
            pad_inches=0.05, facecolor="#FAFBFC")
plt.show()

# Meta regression

In [ ]:
#  META-REGRESSION
#  k = 9 studies
#  Estimator: REML
#  Effect: MD (%)

import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("/content/Data Extraction.csv", encoding="cp1252")

df = df.rename(columns={
    "author":                                     "study",
    "cal_adeq_intervention_percentage":           "med_i",
    "cal_adeq_intervention_percentage_lower_iqr": "q1_i",
    "cal_adeq_intervention_percentage_upper_iqr": "q3_i",
    "cal_adeq_control_percentage":                "med_c",
    "cal_adeq_control_percentage_lower_iqr":      "q1_c",
    "cal_adeq_control_percentage_upper_iqr":      "q3_c",
    "intervention_n":                             "n_i",
    "control_n":                                  "n_c",
    "protocol_type":                              "protocol",
})
df["protocol"] = df["protocol"].str.replace(
    r"[^\x00-\x7F]+", "-", regex=True).str.strip()

def wan_sd(q1, q3, n):
    iqr    = q3 - q1
    p_clip = np.clip((0.75*n - 0.125) / (n + 0.25), 1e-6, 1-1e-6)
    return iqr / (2 * stats.norm.ppf(p_clip))

def wan_mean(q1, med, q3):
    return (q1 + med + q3) / 3

df["mean_i"] = wan_mean(df["q1_i"], df["med_i"], df["q3_i"])
df["mean_c"] = wan_mean(df["q1_c"], df["med_c"], df["q3_c"])
df["sd_i"]   = df.apply(lambda r: wan_sd(r["q1_i"], r["q3_i"], r["n_i"]), axis=1)
df["sd_c"]   = df.apply(lambda r: wan_sd(r["q1_c"], r["q3_c"], r["n_c"]), axis=1)
df["MD"]     = df["mean_i"] - df["mean_c"]
df["var_MD"] = df["sd_i"]**2 / df["n_i"] + df["sd_c"]**2 / df["n_c"]
df["SE_MD"]  = np.sqrt(df["var_MD"])

def reml_tau2(y, v, tau2_init, tol=1e-8, max_iter=1000):
    tau2 = tau2_init if tau2_init > 0 else 0.01
    for _ in range(max_iter):
        w        = 1 / (v + tau2)
        mu       = np.sum(w * y) / np.sum(w)
        score    = -0.5*np.sum(w) + 0.5*np.sum(w**2 * (y - mu)**2)
        info     =  0.5*np.sum(w**2)
        tau2_new = max(0, tau2 + score / info)
        if abs(tau2_new - tau2) < tol:
            return tau2_new
        tau2 = tau2_new
    return tau2

y_all  = df["MD"].values
v_all  = df["var_MD"].values
w_f    = 1 / v_all
mu_f   = np.average(y_all, weights=w_f)
Q_all  = np.sum(w_f * (y_all - mu_f)**2)
k      = len(df)
C_val  = np.sum(w_f) - np.sum(w_f**2) / np.sum(w_f)
tau2_DL    = max(0, (Q_all - (k-1)) / C_val)
tau2_total = reml_tau2(y_all, v_all, tau2_DL)

df["x_protocol"]   = (df["protocol"]            == "Nurse-led protocol").astype(float)
df["x_comparator"] = (df["comparator"]          == "Standard care").astype(float)
df["x_design"]     = (df["study_design"]        == "Interventional").astype(float)
df["x_feed"]       = (df["enteral_feed_route"]  == "Gastric").astype(float)
df["x_age"]        = df["age_median"]    - df["age_median"].mean()
df["x_duration"]   = df["duration_years"]- df["duration_years"].mean()
df["x_logN"]       = np.log(df["total_N"]) - np.log(df["total_N"]).mean()

def metareg(y, v, X_cov, tau2_total, label="", cov_names=None):
    n        = len(y)
    p_count  = X_cov.shape[1]
    df_resid = n - p_count - 1
    if df_resid < 1:
        print(f"  ⚠  {label}: df_resid < 1 — model not estimable")
        return None

    X        = np.column_stack([np.ones(n), X_cov])
    tau2_res = tau2_total

    for _ in range(500):
        w    = 1 / (v + tau2_res)
        W    = np.diag(w)
        XtWX = X.T @ W @ X
        try:
            beta = np.linalg.solve(XtWX, X.T @ W @ y)
        except np.linalg.LinAlgError:
            print(f"  ⚠  {label}: singular matrix")
            return None
        resid    = y - X @ beta
        P        = W - W @ X @ np.linalg.inv(XtWX) @ X.T @ W
        score    = -0.5*np.trace(P) + 0.5*(resid @ P @ P @ resid)
        info     =  0.5*np.trace(P @ P)
        tau2_new = max(0, tau2_res + score / info)
        if abs(tau2_new - tau2_res) < 1e-8:
            tau2_res = tau2_new
            break
        tau2_res = tau2_new

    w        = 1 / (v + tau2_res)
    W        = np.diag(w)
    XtWX     = X.T @ W @ X
    XtWX_inv = np.linalg.inv(XtWX)
    beta     = XtWX_inv @ X.T @ W @ y
    resid    = y - X @ beta

    Q_res  = float(resid @ W @ resid)
    se_KH  = np.sqrt(np.diag((Q_res / df_resid) * XtWX_inv))
    t_vals = beta / se_KH
    p_vals = 2 * (1 - stats.t.cdf(np.abs(t_vals), df=df_resid))
    ci_lo  = beta - stats.t.ppf(0.975, df_resid) * se_KH
    ci_hi  = beta + stats.t.ppf(0.975, df_resid) * se_KH


    R2 = max(0.0, 1.0 - tau2_res / tau2_total) if tau2_total > 0 else np.nan

    w0      = 1 / (v + tau2_total)
    mu0     = np.sum(w0 * y) / np.sum(w0)
    Q_null  = float(np.sum(w0 * (y - mu0)**2))
    Q_model = max(0.0, Q_null - Q_res)
    p_model = 1 - stats.chi2.cdf(Q_model, df=p_count)

    return dict(
        label=label, beta=beta, se=se_KH,
        t=t_vals, p_vals=p_vals, ci_lo=ci_lo, ci_hi=ci_hi,
        tau2_res=tau2_res, R2=R2,
        Q_res=Q_res, df_resid=df_resid,
        Q_model=Q_model, p_model=p_model,
        p_count=p_count, cov_names=cov_names,
    )

def print_metareg(res):
    if res is None:
        return
    names = ["Intercept"] + (res["cov_names"] or [f"X{i}" for i in range(res["p_count"])])
    print(f"\n── Model: {res['label']}")
    print(f"  {'Parameter':<28} {'β':>8} {'SE':>7} {'t':>7} {'p':>7}  {'95% CI'}")
    print("  " + "-"*72)
    for i, name in enumerate(names):
        sig    = " *"  if res['p_vals'][i] < 0.05 else \
                 " †"  if res['p_vals'][i] < 0.10 else ""
        ci_str = f"[{res['ci_lo'][i]:+.2f}, {res['ci_hi'][i]:+.2f}]"
        print(f"  {name:<28} {res['beta'][i]:>+8.3f} {res['se'][i]:>7.3f} "
              f"{res['t'][i]:>+7.3f} {res['p_vals'][i]:>7.4f}  {ci_str}{sig}")
    print(f"  τ²_residual = {res['tau2_res']:.4f}  |  "
          f"R² = {res['R2']*100:.1f}%  |  "
          f"Q_model = {res['Q_model']:.3f} (p={res['p_model']:.4f})  |  "
          f"df_resid = {res['df_resid']}")

y = df["MD"].values
v = df["var_MD"].values


#  UNIVARIABLE MODELS  (7 covariates)

print("\n" + "="*65)
print("  SECTION A — UNIVARIABLE META-REGRESSION  (k=9, df_resid=7)")
print("="*65)

univariable_models = [
    ("Protocol type",           "x_protocol",   ["protocol (Nurse-led=1)"]),
    ("Comparator",              "x_comparator", ["comparator (Std care=1)"]),
    ("Study design",            "x_design",     ["design (Interventional=1)"]),
    ("Enteral feed route",      "x_feed",       ["feed route (Gastric=1)"]),
    ("Age (median, centred)",   "x_age",        ["age_median (centred, months)"]),
    ("Study duration",          "x_duration",   ["duration_years (centred)"]),
    ("Total N (log-centred)",   "x_logN",       ["log(total_N) (centred)"]),
]

uni_results = {}
for label, col, names in univariable_models:
    X_cov             = df[[col]].values
    res               = metareg(y, v, X_cov, tau2_total, label=label, cov_names=names)
    print_metareg(res)
    uni_results[label] = res


#  BIVARIABLE MODELS  (3 pairs)

print("\n\n" + "="*65)
print("  SECTION B — BIVARIABLE META-REGRESSION  (k=9, df_resid=6)")
print("="*65)

print("\n  Pairwise Pearson r among covariates entering models:")
cov_cols      = ["x_protocol","x_comparator","x_design","x_feed",
                 "x_age","x_duration","x_logN"]
corr_df       = df[cov_cols].corr()
pairs_to_check = [
    ("x_protocol",   "x_age",         "Pair 1 — used"),
    ("x_comparator", "x_design",      "Pair 2 — used"),
    ("x_protocol",   "x_duration",    "Pair 3 — used"),
    ("x_protocol",   "x_comparator",  "Excluded (shown for transparency)"),
]
print(f"  {'Pair':<44} {'r':>6}  {'Status / Note'}")
print("  " + "-"*75)
for a, b, note in pairs_to_check:
    r    = corr_df.loc[a, b]
    flag = "  ⚠ HIGH — do not combine" if abs(r) > 0.7 else f"  {note}"
    print(f"  {a:<20} × {b:<22} {r:>+6.3f}{flag}")

bivariable_models = [
    (
        "Protocol + Age (centred)",
        ["x_protocol", "x_age"],
        ["protocol (Nurse-led=1)", "age_median (centred)"],
    ),
    (
        "Comparator + Study design",
        ["x_comparator", "x_design"],
        ["comparator (Std care=1)", "design (Interventional=1)"],
    ),
    (
        "Protocol + Study duration",
        ["x_protocol", "x_duration"],
        ["protocol (Nurse-led=1)", "duration_years (centred)"],
    ),
]

for label, cols, names in bivariable_models:
    X_cov = df[cols].values
    res   = metareg(y, v, X_cov, tau2_total, label=label, cov_names=names)
    print_metareg(res)


#  SUMMARY

print("\n\n" + "="*65)
print("  SECTION C — SUMMARY: ALL UNIVARIABLE MODELS")
print("="*65)
print(f"\n  {'Model':<30} {'β (covariate)':>15} {'p':>8} {'R²(%)':>7} {'τ²_res':>10}")
print("  " + "-"*73)
for label, res in uni_results.items():
    if res is None:
        continue
    b   = res["beta"][1]
    p   = res["p_vals"][1]
    R2  = res["R2"] * 100
    tau = res["tau2_res"]
    sig = " *" if p < 0.05 else (" †" if p < 0.10 else "")
    print(f"  {label:<30} {b:>+15.3f} {p:>8.4f} {R2:>7.1f} {tau:>10.4f}{sig}")

print(f"\n  Overall τ² (intercept-only) = {tau2_total:.4f}")
print("  * p < 0.05  |  † p < 0.10 (trend)")

print(f"""
  ── Interpretation Guide ──────────────────────────────────
  β            = MD change per unit covariate (continuous),
                 or group difference for binary dummies.
  R² analog    = heterogeneity explained; 0% is expected and
                 valid when k=9 (low power for moderation).
  Knapp-Hartung t-test throughout — conservative vs z; CIs
  use t(df_resid), NOT ±1.96 × SE.
  ⚠  All results exploratory with k = 9.
""")

# Sensitivity analysis

In [ ]:
#  SENSITIVITY ANALYSES
#  k = 9 studies | Primary model: RE REML | Effect: MD (%)

import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("/content/Data Extraction.csv", encoding="cp1252")

df = df.rename(columns={
    "author":                                     "study",
    "cal_adeq_intervention_percentage":           "med_i",
    "cal_adeq_intervention_percentage_lower_iqr": "q1_i",
    "cal_adeq_intervention_percentage_upper_iqr": "q3_i",
    "cal_adeq_control_percentage":                "med_c",
    "cal_adeq_control_percentage_lower_iqr":      "q1_c",
    "cal_adeq_control_percentage_upper_iqr":      "q3_c",
    "intervention_n":                             "n_i",
    "control_n":                                  "n_c",
    "protocol_type":                              "protocol",
})
df["protocol"] = df["protocol"].str.replace(
    r"[^\x00-\x7F]+", "-", regex=True).str.strip()

def wan_sd(q1, q3, n):
    iqr    = q3 - q1
    p_clip = np.clip((0.75*n - 0.125) / (n + 0.25), 1e-6, 1-1e-6)
    return iqr / (2 * stats.norm.ppf(p_clip))

def wan_mean(q1, med, q3):
    return (q1 + med + q3) / 3

df["mean_i"] = wan_mean(df["q1_i"], df["med_i"], df["q3_i"])
df["mean_c"] = wan_mean(df["q1_c"], df["med_c"], df["q3_c"])
df["sd_i"]   = df.apply(lambda r: wan_sd(r["q1_i"], r["q3_i"], r["n_i"]), axis=1)
df["sd_c"]   = df.apply(lambda r: wan_sd(r["q1_c"], r["q3_c"], r["n_c"]), axis=1)
df["MD"]     = df["mean_i"] - df["mean_c"]
df["var_MD"] = df["sd_i"]**2 / df["n_i"] + df["sd_c"]**2 / df["n_c"]
df["SE_MD"]  = np.sqrt(df["var_MD"])

def reml_tau2(y, v, tol=1e-8, max_iter=1000):
    """Newton-Raphson REML estimator for τ²."""
    w_f      = 1 / v
    mu_f     = np.average(y, weights=w_f)
    Q        = np.sum(w_f * (y - mu_f)**2)
    k        = len(y)
    C        = np.sum(w_f) - np.sum(w_f**2) / np.sum(w_f)
    tau2     = max(0.01, (Q - (k-1)) / C)
    for _ in range(max_iter):
        w        = 1 / (v + tau2)
        mu       = np.sum(w * y) / np.sum(w)
        score    = -0.5*np.sum(w) + 0.5*np.sum(w**2 * (y - mu)**2)
        info     =  0.5*np.sum(w**2)
        tau2_new = max(0, tau2 + score / info)
        if abs(tau2_new - tau2) < tol:
            return tau2_new
        tau2 = tau2_new
    return tau2

def dl_tau2(y, v):
    """DerSimonian-Laird estimator for τ²."""
    w_f  = 1 / v
    mu_f = np.average(y, weights=w_f)
    Q    = np.sum(w_f * (y - mu_f)**2)
    k    = len(y)
    C    = np.sum(w_f) - np.sum(w_f**2) / np.sum(w_f)
    return max(0, (Q - (k-1)) / C)

def pm_tau2(y, v, tol=1e-8, max_iter=1000):
    """Paule-Mandel iterative method-of-moments estimator."""
    k    = len(y)
    tau2 = 0.0
    for _ in range(max_iter):
        w      = 1 / (v + tau2)
        mu     = np.sum(w * y) / np.sum(w)
        Q      = np.sum(w * (y - mu)**2)
        denom  = np.sum(w) - np.sum(w**2) / np.sum(w)
        tau2_n = max(0, tau2 + (Q - (k-1)) / denom)
        if abs(tau2_n - tau2) < tol:
            return tau2_n
        tau2 = tau2_n
    return tau2

def pool_re(y, v, tau2=None, label=""):
    """Random-effects pooling with given τ² (estimated via REML if None)."""
    k  = len(y)
    if k < 2:
        return None
    t2   = reml_tau2(y, v) if tau2 is None else tau2
    w    = 1 / (v + t2)
    mu   = np.sum(w * y) / np.sum(w)
    se   = np.sqrt(1 / np.sum(w))
    w_f  = 1 / v
    mu_f = np.average(y, weights=w_f)
    Q    = np.sum(w_f * (y - mu_f)**2)
    I2   = max(0, (Q - (k-1)) / Q * 100)
    p_Q  = 1 - stats.chi2.cdf(Q, df=k-1)
    pi_lo = mu - 1.96 * np.sqrt(t2 + se**2)
    pi_hi = mu + 1.96 * np.sqrt(t2 + se**2)
    z    = mu / se
    p    = 2 * (1 - stats.norm.cdf(abs(z)))
    return dict(label=label, k=k, mu=mu, se=se,
                ci_lo=mu-1.96*se, ci_hi=mu+1.96*se,
                tau2=t2, tau=np.sqrt(t2),
                Q=Q, p_Q=p_Q, I2=I2,
                pi_lo=pi_lo, pi_hi=pi_hi, z=z, p=p)

def pool_fe(y, v, label=""):
    """Fixed-effect inverse-variance pooling."""
    w    = 1 / v
    mu   = np.average(y, weights=w)
    se   = np.sqrt(1 / np.sum(w))
    Q    = np.sum(w * (y - mu)**2)
    k    = len(y)
    I2   = max(0, (Q - (k-1)) / Q * 100)
    p_Q  = 1 - stats.chi2.cdf(Q, df=k-1)
    z    = mu / se
    p    = 2 * (1 - stats.norm.cdf(abs(z)))
    return dict(label=label, k=k, mu=mu, se=se,
                ci_lo=mu-1.96*se, ci_hi=mu+1.96*se,
                Q=Q, p_Q=p_Q, I2=I2,
                tau2=0.0, tau=0.0,
                pi_lo=np.nan, pi_hi=np.nan, z=z, p=p)

def fmt_res(res, show_pi=False, show_tau=True):
    tau_str = f"  τ²={res['tau2']:.2f}" if show_tau else ""
    pi_str  = f"  PI=[{res['pi_lo']:.2f}, {res['pi_hi']:.2f}]" if show_pi else ""
    print(f"  k={res['k']}  MD={res['mu']:+.2f}%  "
          f"95%CI=[{res['ci_lo']:.2f}, {res['ci_hi']:.2f}]  "
          f"p={res['p']:.4f}  I²={res['I2']:.1f}%{tau_str}{pi_str}")

y_all        = df["MD"].values
v_all        = df["var_MD"].values
tau2_primary = reml_tau2(y_all, v_all)
primary      = pool_re(y_all, v_all, tau2=tau2_primary,
                       label="Primary (RE REML, all 9 studies)")

print("=" * 65)
print("  SECTION 7 — SENSITIVITY ANALYSES")
print("=" * 65)
print(f"\n  PRIMARY REFERENCE (RE REML, k=9):")
fmt_res(primary, show_pi=True)

print("\n\n" + "="*65)
print("  SA1 — LEAVE-ONE-OUT ANALYSIS")
print("="*65)
print(f"\n  Primary: MD=+{primary['mu']:.2f}%  I²={primary['I2']:.1f}%  τ²={tau2_primary:.2f}\n")
print(f"  {'Study omitted':<26} {'MD':>7} {'95% CI':>22} "
      f"{'p':>7} {'I²':>6} {'τ²':>9}  {'ΔMD':>7}")
print("  " + "-"*85)

loo_results = []
for i, row in df.iterrows():
    mask  = df.index != i
    y_loo = df.loc[mask, "MD"].values
    v_loo = df.loc[mask, "var_MD"].values
    res   = pool_re(y_loo, v_loo)
    if res is None:
        continue
    delta = res["mu"] - primary["mu"]
    loo_results.append({**res, "omitted": row["study"], "delta": delta})
    flag  = "  ◄ notable" if (abs(delta) > 5 or res["I2"] < 50) else ""
    print(f"  {row['study']:<26} {res['mu']:>+7.2f} "
          f"[{res['ci_lo']:>+6.2f}, {res['ci_hi']:>+6.2f}] "
          f"{res['p']:>7.4f} {res['I2']:>5.1f}% {res['tau2']:>9.2f} "
          f"{delta:>+7.2f}{flag}")

max_delta  = max(loo_results, key=lambda x: abs(x["delta"]))
min_I2_loo = min(loo_results, key=lambda x: x["I2"])
all_pos    = all(r["mu"] > 0   for r in loo_results)
all_sig    = all(r["p"]  < 0.05 for r in loo_results)

print(f"\n  ► Largest MD shift: omitting {max_delta['omitted']} "
      f"(ΔMD = {max_delta['delta']:+.2f}%)")
print(f"  ► Largest I² reduction: omitting {min_I2_loo['omitted']} "
      f"(I² → {min_I2_loo['I2']:.1f}%)")
print(f"  ► Positive direction consistent across all LOO models: "
      + ("YES ✓" if all_pos else "NO ✗"))
print(f"  ► All LOO estimates statistically significant (p<0.05): "
      + ("YES ✓" if all_sig else "NO ✗ — see table"))

print("\n\n" + "="*65)
print("  SA2 — FIXED-EFFECT vs RANDOM-EFFECTS")
print("="*65)

fe = pool_fe(y_all, v_all, label="Fixed-Effect (IV)")
re = pool_re(y_all, v_all, tau2=tau2_primary, label="Random-Effects (REML)")

print(f"\n  {'Model':<30} {'MD':>7} {'95% CI':>22} {'SE':>6} {'p':>7} {'I²':>6}")
print("  " + "-"*78)
for res in [re, fe]:
    print(f"  {res['label']:<30} {res['mu']:>+7.2f} "
          f"[{res['ci_lo']:>+6.2f}, {res['ci_hi']:>+6.2f}] "
          f"{res['se']:>6.3f} {res['p']:>7.4f} {res['I2']:>5.1f}%")

print(f"\n  ΔMD (FE − RE)  = {fe['mu']-re['mu']:+.2f}%")
print(f"  CI width: FE = {fe['ci_hi']-fe['ci_lo']:.2f}%  |  "
      f"RE = {re['ci_hi']-re['ci_lo']:.2f}%")
print(f"\n  ► RE model is the appropriate primary model (I²=73.9%).")
print(f"    FE result reported as a precision-bound sensitivity check only.")

print("\n\n" + "="*65)
print("  SA3 — INFLUENCE ANALYSIS")
print("  (Studentised deleted residuals | Cook's D | DFFITS)")
print("="*65)

k_all = len(df)
w_re  = 1 / (v_all + tau2_primary)
mu_re = np.sum(w_re * y_all) / np.sum(w_re)
se_re = np.sqrt(1 / np.sum(w_re))

print(f"\n  {'Study':<26} {'MD':>7} {'RawResid':>10} {'StudResid':>11}"
      f"{'Cook\'sD':>9} {'DFFITS':>9} {'Flag'}")
print("  " + "-"*88)

influence_results = []
for i, row in df.iterrows():
    mask  = df.index != i
    y_loo = df.loc[mask, "MD"].values
    v_loo = df.loc[mask, "var_MD"].values
    res_i = pool_re(y_loo, v_loo)
    if res_i is None:
        continue

    vi_i       = v_all[list(df.index).index(i)]
    raw_resid  = row["MD"] - mu_re
    var_del    = vi_i + res_i["tau2"] + res_i["se"]**2
    stud_resid = (row["MD"] - res_i["mu"]) / np.sqrt(var_del)
    dffits     = (mu_re - res_i["mu"]) / se_re
    cooks_d    = (mu_re - res_i["mu"])**2 / (se_re**2)

    flags = []
    if abs(stud_resid) > 1.96:      flags.append("outlier")
    if abs(dffits)     > 1.0:       flags.append("influential")
    if cooks_d         > 4/k_all:   flags.append("Cook>4/k")
    flag_str = ", ".join(flags) if flags else "—"

    influence_results.append({
        "study": row["study"], "stud_resid": stud_resid,
        "cooks_d": cooks_d, "dffits": dffits, "flags": flag_str
    })
    print(f"  {row['study']:<26} {row['MD']:>+7.2f} {raw_resid:>+10.3f} "
          f"{stud_resid:>+11.3f} {cooks_d:>9.4f} {dffits:>+9.4f}  {flag_str}")

print(f"\n  Thresholds:  |StudentisedResid| > 1.96 → outlier")
print(f"               |DFFITS| > 1.0 → influential")
print(f"               Cook's D > 4/k = {4/k_all:.3f} → influential")

flagged = [r for r in influence_results if r["flags"] != "—"]
if flagged:
    print(f"\n  ► Flagged studies:")
    for r in flagged:
        print(f"    • {r['study']}: {r['flags']}")
else:
    print(f"\n  ► No studies flagged on any influence metric.")

print("\n\n" + "="*65)
print("  SA4 — RESTRICTION: INTERVENTIONAL STUDIES ONLY (k=4)")
print("="*65)

df_int = df[df["study_design"] == "Interventional"]
y_int  = df_int["MD"].values
v_int  = df_int["var_MD"].values

print(f"\n  Studies: {df_int['study'].tolist()}")
res_int = pool_re(y_int, v_int)
fmt_res(res_int, show_pi=True)
print(f"\n  ΔMD vs primary = {res_int['mu']-primary['mu']:+.2f}%")
print(f"  I²: {primary['I2']:.1f}% → {res_int['I2']:.1f}%  |  "
      f"τ²: {tau2_primary:.2f} → {res_int['tau2']:.2f}")
print(f"  ► Significant: {'YES ✓' if res_int['p'] < 0.05 else 'NO ✗'} "
      f"(p={res_int['p']:.4f})")

print("\n\n" + "="*65)
print("  SA5 — RESTRICTION: OBSERVATIONAL STUDIES ONLY (k=5)")
print("="*65)

df_obs = df[df["study_design"] == "Observational"]
y_obs  = df_obs["MD"].values
v_obs  = df_obs["var_MD"].values

print(f"\n  Studies: {df_obs['study'].tolist()}")
res_obs = pool_re(y_obs, v_obs)
fmt_res(res_obs, show_pi=True)
print(f"\n  ΔMD vs primary = {res_obs['mu']-primary['mu']:+.2f}%")
print(f"  I²: {primary['I2']:.1f}% → {res_obs['I2']:.1f}%  |  "
      f"τ²: {tau2_primary:.2f} → {res_obs['tau2']:.2f}")
print(f"  ► Significant: {'YES ✓' if res_obs['p'] < 0.05 else 'NO ✗'} "
      f"(p={res_obs['p']:.4f})")

print("\n\n" + "="*65)
print("  SA6 — ESTIMATOR SENSITIVITY: REML vs DL vs Paule-Mandel")
print("="*65)

tau2_reml = reml_tau2(y_all, v_all)
tau2_dl   = dl_tau2(y_all, v_all)
tau2_pm   = pm_tau2(y_all, v_all)

estimators = [
    ("REML (primary)", tau2_reml),
    ("DerSimonian-Laird", tau2_dl),
    ("Paule-Mandel", tau2_pm),
]

print(f"\n  {'Estimator':<24} {'τ²':>9} {'τ':>7} {'MD':>7} "
      f"{'95% CI':>22} {'p':>7} {'I²':>6}")
print("  " + "-"*82)

est_results = {}
for est_label, t2 in estimators:
    res = pool_re(y_all, v_all, tau2=t2)
    est_results[est_label] = res
    print(f"  {est_label:<24} {t2:>9.4f} {np.sqrt(t2):>7.4f} "
          f"{res['mu']:>+7.2f} [{res['ci_lo']:>+6.2f}, {res['ci_hi']:>+6.2f}] "
          f"{res['p']:>7.4f} {res['I2']:>5.1f}%")

tau_vals = [tau2_reml, tau2_dl, tau2_pm]
ci_widths = [est_results[l]["ci_hi"] - est_results[l]["ci_lo"]
             for l, _ in estimators]
all_sig_est = all(est_results[l]["p"] < 0.05 for l, _ in estimators)
print(f"\n  τ² range:  {min(tau_vals):.2f} – {max(tau_vals):.2f}")
print(f"  CI width range: {min(ci_widths):.2f}% – {max(ci_widths):.2f}%")
print(f"  All estimators significant (p<0.05): "
      + ("YES ✓" if all_sig_est else "NO ✗"))

res_dl_full = est_results["DerSimonian-Laird"]
res_pm_full = est_results["Paule-Mandel"]

print("\n\n" + "="*65)
print("  SA7 — RESTRICTION: STANDARD CARE COMPARATOR ONLY (k=5)")
print("="*65)

df_sc  = df[df["comparator"] == "Standard care"]
y_sc   = df_sc["MD"].values
v_sc   = df_sc["var_MD"].values

print(f"\n  Studies: {df_sc['study'].tolist()}")
res_sc = pool_re(y_sc, v_sc)
fmt_res(res_sc, show_pi=True)
print(f"\n  ΔMD vs primary = {res_sc['mu']-primary['mu']:+.2f}%")
print(f"  I²: {primary['I2']:.1f}% → {res_sc['I2']:.1f}%  |  "
      f"τ²: {tau2_primary:.2f} → {res_sc['tau2']:.2f}")
print(f"  ► Significant: {'YES ✓' if res_sc['p'] < 0.05 else 'NO ✗'} "
      f"(p={res_sc['p']:.4f})")

print("\n\n" + "="*65)
print("  CONSOLIDATED SENSITIVITY ANALYSIS SUMMARY")
print("="*65)

summary_rows = [
    ("Primary — RE REML (k=9)",         primary),
    ("SA2 — Fixed-Effect (k=9)",         fe),
    ("SA4 — Interventional only (k=4)",  res_int),
    ("SA5 — Observational only (k=5)",   res_obs),
    ("SA6 — DL estimator (k=9)",         res_dl_full),
    ("SA6 — Paule-Mandel (k=9)",         res_pm_full),
    ("SA7 — Standard care only (k=5)",   res_sc),
]

print(f"\n  {'Analysis':<38} {'k':>3} {'MD':>7} {'95% CI':>22} "
      f"{'p':>7} {'I²':>6}")
print("  " + "-"*85)
for label, res in summary_rows:
    if res is None:
        continue
    sig = " *" if res["p"] < 0.05 else ""
    print(f"  {label:<38} {res['k']:>3} {res['mu']:>+7.2f} "
          f"[{res['ci_lo']:>+6.2f}, {res['ci_hi']:>+6.2f}] "
          f"{res['p']:>7.4f} {res['I2']:>5.1f}%{sig}")

mus  = [r["mu"] for _, r in summary_rows if r is not None]
sigs = [r["p"] < 0.05 for _, r in summary_rows if r is not None]
dirs = [r["mu"] > 0   for _, r in summary_rows if r is not None]

print(f"\n  * p < 0.05")
print(f"\n  ── Robustness Summary ────────────────────────────────")
print(f"  MD range across all analyses : {min(mus):+.2f}% to {max(mus):+.2f}%")
print(f"  Consistent positive direction: {'YES ✓' if all(dirs) else 'NO ✗'}")
print(f"  Significant in {sum(sigs)}/{len(sigs)} analyses")

In [ ]:
import sys
print(sys.version)